# Figure 9 — Classical NJ with subsampling on a synthetic unbalanced (birth–death) tree

Re-runs the Figure-8 experiment but with `tree_model='birth_death'` (birth_rate=1,
death_rate=0) instead of `balanced_binary`. Motivation: test whether the
user's intuition ("more samples at fixed $p$ should average down noise")
shows up when the tree topology is **unbalanced** (so the distance matrix
is heterogeneous) but **n-comparable** (so NJ can recover the tree at
$p=1$ across all $n$).

Earlier attempts with `kingman` failed because Kingman tip-branch lengths
shrink as $O(1/n^2)$, so NJ couldn't recover the truth at $p=1$ for
$n \geq 256$ — the experiment was confounded by topology resolution, not
by subsampling. Birth–death with constant birth rate keeps branch lengths
$O(1)$ regardless of $n$; nRF at $p=1$ stays at $\sim 3\%$ for all
$n \in \{128, 256, 512, 1024, 2048\}$.

Four panels, same layout as Figure 8: raw spec-norm, nRF, $L_\infty$,
relative spec-norm. The focused diagnostic is bottom-right: does
`rel_specnorm` separate by $n$, or still collapse onto $(1-p)/p$ as it did
for balanced-binary?

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, json
from datetime import datetime
from pathlib import Path
from IPython.display import Image, display


_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

# Drop any stale cached copies of our project modules so this cell picks up
# edits made since the kernel started. autoreload handles subsequent edits.
for _mod in [m for m in list(sys.modules) if m.startswith(('scripts.', 'src.'))]:
    del sys.modules[_mod]

from src.config.presets import custom_config
from src.runners.nj_sweep import nj_sweep_for_params
from src.utils.nj_io import load_nj_results
from scripts.plot_nj_three_panel import _plot_three_panel
from scripts.plot_nj_p_star_vs_n import (
    _load_sweep, _compute_p_stars, _plot,
    _plot_overlay, _plot_q_overlay,
)

## Choose: load or launch
Set `RUN_DIR` to an existing sweep directory to **load** prior results. Leave it `None` to **launch** a fresh sweep with the grid below.

In [ ]:
# Default: load the existing birth_death L=10000 sweep. Set RUN_DIR=None to
# launch a fresh sweep with the TAXA / SEQLEN / REPS / P_VALS grid below.
RUN_DIR = Path(_root) / 'results' / 'runs' / '20260519-224641-nj_sweep_birth_death_L10000'

TAXA   = [128, 256, 512, 1024, 2048]
SEQLEN = 10000
REPS   = 5
TREE_MODEL = 'birth_death'
P_VALS = [1.0, 0.9, 0.5, 0.1, 0.05, 0.01, 0.005, 0.001, 0.0005, 0.0001]

if RUN_DIR is None:
    RUN_DIR = Path(_root) / 'results' / 'runs' / (datetime.now().strftime('%Y%m%d-%H%M%S')
                                          + '-nj_sweep_' + TREE_MODEL + '_L' + str(SEQLEN))
    for n in TAXA:
        cfg = custom_config(num_taxa=n, sequence_length=SEQLEN, mutation_rate=0.1,
                            tree_model=TREE_MODEL, seq_model='JC69',
                            p_values=P_VALS, bootstrap_reps=REPS,
                            sampling_method='uniform', matrix_kind='distance',
                            birth_rate=1.0, death_rate=0.0)
        nj_sweep_for_params(cfg, n, SEQLEN, str(RUN_DIR / f'n{n}_L{SEQLEN}'))
else:
    RUN_DIR = Path(RUN_DIR)
print('sweep dir:', RUN_DIR)

## Three-panel plot, per n

In [ ]:
# from IPython.display import Image, display
# for sub in sorted(RUN_DIR.iterdir()):
#     if not sub.is_dir():
#         continue
#     if not ((sub / 'nj_meta.json').exists() or (sub / 'nj_results.json').exists()):
#         continue
#     r = load_nj_results(sub)
#     out = sub / 'nj_three_panel.png'
#     _plot_three_panel(r, out)
#     display(Image(filename=str(out)))

## p* vs n

In [ ]:
rows = _compute_p_stars(_load_sweep(RUN_DIR))

# Diagnostic-metric hint: the 2x2 overlay panel renders normalised metrics
# (L_inf and ||D - D_hat||_2 / ||D||_2) only when nj_meta_extra.json exists
# alongside each nj_meta.json. If none are found, _plot_overlay falls back to
# the legacy 1x2 layout — print the command to generate the extras.
_has_extras = any(p.exists() for p in RUN_DIR.glob('n*_L*/nj_meta_extra.json'))
if not _has_extras:
    print(f"[hint] no nj_meta_extra.json under {RUN_DIR}; the overlay will "
          f"render the legacy 1x2 layout. To add the normalised diagnostic "
          f"panels (L_inf and ||D - D_hat||_2 / ||D||_2), run:\n"
          f"    python scripts/nj_recompute_normalized_metrics.py {RUN_DIR}\n")

_plot(rows, RUN_DIR / 'nj_p_star_vs_n.png')
_plot_overlay(RUN_DIR, RUN_DIR / 'nj_overlay_specnorm_rf.png')
_plot_q_overlay(RUN_DIR, RUN_DIR / 'nj_overlay_q.png')

for name in ('nj_overlay_specnorm_rf.png', 'nj_overlay_q.png', 'nj_p_star_vs_n.png'):
    display(Image(filename=str(RUN_DIR / name)))
rows